# BERT (Colab) — Training Size Sweep (Exp 06)

Iterative benchmark sweep over training set sizes using the best configuration identified in Exp 05.

**Why this config (carried over from Exp 05):**
- `lr=1.5e-5`: monotonically best for threat F1 and macro F1 across all 16 Exp 04 sweep runs
- `weight_decay=0.015`: marginally but consistently better than 0.010
- `warmup_ratio=0.06`: best macro F1 in sweep; shorter warmup is well-suited to small dataset sizes too
- `max_length=192`: BERT's +0.0914 threat F1 advantage over DistilBERT is attributed to longer context; keeping it ensures the sweep isolates data-volume effects rather than context effects
- `batch_size=16` + `grad_accum=2` → effective 32: same effective batch as DistilBERT final sweep; smaller per-step batch required due to BERT's 109M params vs 67M
- `BCEWithLogitsLoss` no pos_weight: confirmed best across all 6 loss variants in Exp D+E
- Threshold grid `0.05–0.995, step=0.005`: finer grid from Exp 05; rare-label optima cluster at low thresholds where 0.01-step grids lose resolution

**What this sweep adds over Exp 05:**
- Isolates BERT's data-efficiency curve: how much data does BERT actually need to match its full-dataset performance?
- Compares BERT's scaling behaviour against the DistilBERT final sweep at matching train sizes
- Identifies at what training size BERT's threat-label advantage first emerges
- Checks whether BERT's threshold calibration stabilises faster or slower than DistilBERT's as data grows

**Fixed config:**
- model: `bert-base-uncased`, `problem_type='multi_label_classification'`
- data split: `validation_fraction=0.1`, `random_state=42`, `use_iterative_stratify=True`, `rebalance_train=False`
- tokenization: `max_length=192`, `padding='max_length'`, `truncation=True`
- training: `batch_size=16`, `gradient_accumulation_steps=2` (effective 32), `max_epochs=5`
- optimization: AdamW, `lr=1.5e-5`, `weight_decay=0.015`, linear warmup+decay, `warmup_ratio=0.06`
- regularization: early stopping on val loss (`patience=2`, `min_delta=0.0`, restore best weights)
- loss: `BCEWithLogitsLoss`, `pos_weight=None`
- precision: bf16 AMP if supported
- evaluation: per-label tuned thresholds on `np.arange(0.05, 0.996, 0.005)` (0.005 step)
- reproducibility: seed=42

In [ ]:
!pip -q install torch transformers pandas numpy matplotlib scikit-learn seaborn iterative-stratification

In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ===== CHANGE THIS to your repo folder in Drive =====
PROJECT_ROOT = Path('/content/drive/MyDrive/cmpe258/jigsaw-toxic-comment-classification-challenge')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'PROJECT_ROOT not found: {PROJECT_ROOT}. Upload/clone your repo to this path first.'
    )

NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
ARTIFACT_DIR  = NOTEBOOKS_DIR / 'bert' / 'bert_06'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOKS_DIR))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('ARTIFACT_DIR:', ARTIFACT_DIR)

In [ ]:
import contextlib
import math
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import confusion_matrix, f1_score, roc_curve, auc

from preprocessing.text_preprocessing import LABEL_COLUMNS, preprocess_for_bert
from metrics_helpers import multilabel_evaluation_report, torch_parameter_count

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

RARE_LABELS   = ['severe_toxic', 'threat', 'identity_hate']
COMMON_LABELS = ['toxic', 'obscene', 'insult']
LABEL_COLORS  = {
    'toxic':         '#e74c3c',
    'severe_toxic':  '#c0392b',
    'obscene':       '#e67e22',
    'threat':        '#8e44ad',
    'insult':        '#2980b9',
    'identity_hate': '#16a085',
}

# DistilBERT final-sweep reference data (embedded from artifacts_colab_final_train_size_sweep)
DISTILBERT_MACRO_F1 = {
    10000: 0.5444, 20000: 0.6312, 30000: 0.6200,  40000: 0.6603,
    50000: 0.6786, 60000: 0.6765, 70000: 0.6730,  80000: 0.6891,
    90000: 0.6982, 100000: 0.6911, 110000: 0.6939, 120000: 0.6892,
    130000: 0.6910, 140000: 0.7041, 143613: 0.6960,
}
DISTILBERT_MICRO_F1 = {
    10000: 0.7261, 20000: 0.7752, 30000: 0.7697,  40000: 0.7824,
    50000: 0.7852, 60000: 0.7837, 70000: 0.7843,  80000: 0.7822,
    90000: 0.7904, 100000: 0.7860, 110000: 0.7830, 120000: 0.7899,
    130000: 0.7976, 140000: 0.7954, 143613: 0.7950,
}
DISTILBERT_THREAT_F1 = {
    10000: 0.0836, 20000: 0.3462, 30000: 0.3400,  40000: 0.4510,
    50000: 0.5217, 60000: 0.5432, 70000: 0.4719,  80000: 0.5714,
    90000: 0.5918, 100000: 0.5946, 110000: 0.5854, 120000: 0.5686,
    130000: 0.5495, 140000: 0.5926, 143613: 0.5679,
}
DISTILBERT_TRAIN_TIME = {
    10000: 97, 20000: 195, 30000: 293, 40000: 390,
    50000: 391, 60000: 469, 70000: 547, 80000: 625,
    90000: 703, 100000: 780, 110000: 859, 120000: 938,
    130000: 1269, 140000: 1094, 143613: 1403,
}
DISTILBERT_FULL_F1 = {
    'toxic': 0.8408, 'severe_toxic': 0.5431, 'obscene': 0.8445,
    'threat': 0.5679, 'insult': 0.7718, 'identity_hate': 0.6082,
}
DISTILBERT_FULL_ROC = {
    'toxic': 0.9869, 'severe_toxic': 0.9917, 'obscene': 0.9923,
    'threat': 0.9861, 'insult': 0.9890, 'identity_hate': 0.9880,
}


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def binary_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    denom = 2 * tp + fp + fn
    return float((2 * tp) / denom) if denom > 0 else 0.0


def tune_per_label_thresholds(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    labels,
    threshold_grid: np.ndarray,
):
    best_thresholds = {}
    rows = []
    for j, label in enumerate(labels):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]
        best_t, best_f1 = 0.5, -1.0
        for t in threshold_grid:
            y_pred_j = (y_prob_j >= t).astype(np.int64)
            f1_j = binary_f1(y_true_j, y_pred_j)
            if f1_j > best_f1:
                best_f1 = f1_j
                best_t  = float(t)
        best_thresholds[label] = best_t
        rows.append({'label': label, 'best_threshold': round(best_t, 4), 'best_f1_on_val': round(best_f1, 6)})
    return best_thresholds, pd.DataFrame(rows)


def apply_thresholds(y_prob: np.ndarray, labels, thresholds: dict) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=np.int64)
    for j, label in enumerate(labels):
        out[:, j] = (y_prob[:, j] >= thresholds[label]).astype(np.int64)
    return out


def make_confusion_artifacts(y_true: np.ndarray, y_pred: np.ndarray, labels):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    per_label_rows = []
    for j, label in enumerate(labels):
        cm = confusion_matrix(y_true[:, j], y_pred[:, j], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        per_label_rows.append({'label': label, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)})
    per_label_df = pd.DataFrame(per_label_rows)
    agg_cm = confusion_matrix(y_true.ravel(), y_pred.ravel(), labels=[0, 1])
    agg_tn, agg_fp, agg_fn, agg_tp = agg_cm.ravel()
    aggregate_df = pd.DataFrame([{'tn': int(agg_tn), 'fp': int(agg_fp), 'fn': int(agg_fn), 'tp': int(agg_tp)}])
    return per_label_df, aggregate_df


class EncDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc    = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item


def collate(batch):
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0] if k != 'labels'}
    out['labels'] = torch.stack([b['labels'] for b in batch], dim=0)
    return out


def enc_dict(enc):
    # BERT produces input_ids, attention_mask, and token_type_ids
    keys = [k for k in ('input_ids', 'attention_mask', 'token_type_ids') if k in enc]
    return {k: enc[k] for k in keys}

In [ ]:
# ── Configuration (Exp 05 best config, fixed for sweep) ─────────────────────
DEVICE = pick_device()
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True

torch.manual_seed(42)
np.random.seed(42)

# model
MODEL_NAME   = 'bert-base-uncased'
PROBLEM_TYPE = 'multi_label_classification'

# data
VALIDATION_FRACTION    = 0.1
RANDOM_STATE           = 42
USE_ITERATIVE_STRATIFY = True
REBALANCE_TRAIN        = False

# tokenization — 192 keeps BERT's long-context advantage on threat
MAX_LENGTH = 192

# training
BATCH_SIZE                  = 16
GRADIENT_ACCUMULATION_STEPS = 2   # effective batch = 32
MAX_EPOCHS                  = 5
LR                          = 1.5e-5
WEIGHT_DECAY                = 0.015
WARMUP_RATIO                = 0.06
MAX_GRAD_NORM               = 1.0

# regularization
EARLY_STOP_ENABLED   = True
EARLY_STOP_PATIENCE  = 2
EARLY_STOP_MIN_DELTA = 0.0

# loss
USE_POS_WEIGHT = False

# precision
USE_BF16          = True
USE_TORCH_COMPILE = False
NUM_WORKERS       = 2

# evaluation — finer 0.005-step grid from Exp 05
THRESHOLD_GRID = np.arange(0.05, 0.996, 0.005)

# sweep
SIZE_STEP       = 10_000
MAX_VAL_SAMPLES = None
SKIP_COMPLETED  = True   # resume after Colab disconnect

# reproducibility
TORCH_SEED  = 42
NUMPY_SEED  = 42

_bf16_ok = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
USE_AMP   = bool(USE_BF16 and _bf16_ok)
if USE_BF16 and DEVICE.type == 'cuda' and not _bf16_ok:
    print('USE_BF16 requested but bf16 not supported on this GPU; running in fp32.')

print(f'Device:          {DEVICE}')
print(f'AMP (bf16):      {USE_AMP}')
print(f'Model:           {MODEL_NAME}')
print(f'Max length:      {MAX_LENGTH}')
print(f'LR:              {LR}')
print(f'Weight decay:    {WEIGHT_DECAY}')
print(f'Warmup ratio:    {WARMUP_RATIO}')
print(f'Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}')
print(f'Threshold steps: {len(THRESHOLD_GRID)} ({THRESHOLD_GRID[0]:.3f}\u2013{THRESHOLD_GRID[-1]:.3f}, step={THRESHOLD_GRID[1]-THRESHOLD_GRID[0]:.3f})')

In [ ]:
# ── Main training-size sweep ─────────────────────────────────────────────────
summary_rows              = []
per_label_frames          = []
threshold_frames          = []
conf_per_label_base_frames  = []
conf_per_label_tuned_frames = []
conf_agg_base_rows          = []
conf_agg_tuned_rows         = []

pin = DEVICE.type == 'cuda'

torch.manual_seed(TORCH_SEED)
np.random.seed(NUMPY_SEED)

# Probe full dataset size after the validation split
print('Probing full dataset size...')
_probe = preprocess_for_bert(
    validation_fraction=VALIDATION_FRACTION,
    random_state=RANDOM_STATE,
    max_length=MAX_LENGTH,
    return_tensors='pt',
    max_train_samples=None,
    max_val_samples=MAX_VAL_SAMPLES,
    use_iterative_stratify=USE_ITERATIVE_STRATIFY,
    rebalance_train=REBALANCE_TRAIN,
    print_diagnostics=True,
)
full_train_count = int(np.asarray(_probe.y_train).shape[0])
print(f'Full train rows after split: {full_train_count}')
del _probe

# Build sweep size list: 10k, 20k, ..., full
if full_train_count <= SIZE_STEP:
    train_sizes = [full_train_count]
else:
    train_sizes = list(range(SIZE_STEP, full_train_count + 1, SIZE_STEP))
    if train_sizes[-1] != full_train_count:
        train_sizes.append(full_train_count)

run_specs = [
    {'train_size': int(s), 'train_fraction': float(s) / float(full_train_count), 'max_train_samples': int(s)}
    for s in train_sizes
]
print(f'Sweep sizes: {train_sizes[:4]} ... {train_sizes[-1]}  (n={len(train_sizes)})')

# ── Per-run loop ─────────────────────────────────────────────────────────────
for idx, spec in enumerate(run_specs, start=1):
    run_id = f"train_size_{spec['train_size']}"

    row_file                = ARTIFACT_DIR / f'summary_{run_id}.csv'
    per_label_file          = ARTIFACT_DIR / f'per_label_{run_id}.csv'
    thresholds_file         = ARTIFACT_DIR / f'thresholds_{run_id}.csv'
    conf_pl_base_file       = ARTIFACT_DIR / f'confusion_per_label_baseline_{run_id}.csv'
    conf_pl_tuned_file      = ARTIFACT_DIR / f'confusion_per_label_tuned_{run_id}.csv'
    conf_agg_base_file      = ARTIFACT_DIR / f'confusion_aggregate_baseline_{run_id}.csv'
    conf_agg_tuned_file     = ARTIFACT_DIR / f'confusion_aggregate_tuned_{run_id}.csv'

    files_exist = all([
        row_file.exists(), per_label_file.exists(), thresholds_file.exists(),
        conf_pl_base_file.exists(), conf_pl_tuned_file.exists(),
        conf_agg_base_file.exists(), conf_agg_tuned_file.exists(),
    ])

    if SKIP_COMPLETED and files_exist:
        try:
            print(f'[{idx}/{len(run_specs)}] {run_id}: skipping (already saved)')
            summary_rows.append(pd.read_csv(row_file).iloc[0].to_dict())
            per_label_frames.append(pd.read_csv(per_label_file))
            threshold_frames.append(pd.read_csv(thresholds_file))
            conf_per_label_base_frames.append(pd.read_csv(conf_pl_base_file))
            conf_per_label_tuned_frames.append(pd.read_csv(conf_pl_tuned_file))
            conf_agg_base_rows.append(pd.read_csv(conf_agg_base_file).iloc[0].to_dict())
            conf_agg_tuned_rows.append(pd.read_csv(conf_agg_tuned_file).iloc[0].to_dict())
            continue
        except OSError as e:
            print(f'[{idx}/{len(run_specs)}] {run_id}: Drive read failed ({e}); rerunning.')

    print(
        f"[{idx}/{len(run_specs)}] {run_id}: running "
        f"(max_train={spec['max_train_samples']}, frac={spec['train_fraction']:.4f})"
    )

    # ── Data loading ─────────────────────────────────────────────────────────
    run_data = preprocess_for_bert(
        validation_fraction=VALIDATION_FRACTION,
        random_state=RANDOM_STATE,
        max_length=MAX_LENGTH,
        return_tensors='pt',
        max_train_samples=spec['max_train_samples'],
        max_val_samples=MAX_VAL_SAMPLES,
        use_iterative_stratify=USE_ITERATIVE_STRATIFY,
        rebalance_train=REBALANCE_TRAIN,
        print_diagnostics=False,
    )

    train_enc = enc_dict(run_data.train_encodings)
    val_enc   = enc_dict(run_data.val_encodings)
    y_train   = np.asarray(run_data.y_train, dtype=np.float32)
    y_val     = np.asarray(run_data.y_val,   dtype=np.int64)

    train_loader = DataLoader(
        EncDataset(train_enc, y_train),
        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
        num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=NUM_WORKERS > 0,
    )
    val_loader = DataLoader(
        EncDataset(val_enc, y_val),
        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate,
        num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=NUM_WORKERS > 0,
    )

    # ── Model ────────────────────────────────────────────────────────────────
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABEL_COLUMNS), problem_type=PROBLEM_TYPE,
    ).to(DEVICE)
    num_params = torch_parameter_count(model)

    # ── Optimizer + scheduler ────────────────────────────────────────────────
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer = torch.optim.AdamW(
        [
            {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
             'weight_decay': WEIGHT_DECAY},
            {'params': [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)],
             'weight_decay': 0.0},
        ],
        lr=LR,
    )

    steps_per_epoch    = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
    num_training_steps = steps_per_epoch * MAX_EPOCHS
    warmup_steps       = max(0, min(int(WARMUP_RATIO * num_training_steps), num_training_steps - 1))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps
    )

    autocast_ctx = (
        torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=True)
        if USE_AMP else contextlib.nullcontext()
    )

    # ── Training loop ────────────────────────────────────────────────────────
    best_val_loss   = float('inf')
    best_state_cpu  = None
    best_epoch      = 0
    patience_left   = EARLY_STOP_PATIENCE
    epochs_ran      = 0
    early_stopped   = False
    train_time_s    = 0.0
    inference_time_s = 0.0
    train_loss_last = float('nan')
    val_loss_last   = float('nan')

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss_sum = 0.0
        epoch_batches  = 0
        t0   = time.perf_counter()
        micro = 0
        optimizer.zero_grad(set_to_none=True)

        for batch in train_loader:
            batch  = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
            labels = batch.pop('labels')
            with autocast_ctx:
                logits = model(**batch).logits
                loss   = F.binary_cross_entropy_with_logits(logits, labels) / GRADIENT_ACCUMULATION_STEPS
            loss.backward()
            micro          += 1
            epoch_loss_sum += float(loss.item()) * GRADIENT_ACCUMULATION_STEPS
            epoch_batches  += 1

            if micro % GRADIENT_ACCUMULATION_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        if micro % GRADIENT_ACCUMULATION_STEPS != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        train_time_s   += time.perf_counter() - t0
        train_loss_last = epoch_loss_sum / max(epoch_batches, 1)

        # ── Validation ───────────────────────────────────────────────────────
        model.eval()
        val_loss_sum  = 0.0
        val_batches   = 0
        t1 = time.perf_counter()
        with torch.no_grad():
            for batch in val_loader:
                batch  = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
                labels = batch.pop('labels')
                with autocast_ctx:
                    logits = model(**batch).logits
                    vloss  = F.binary_cross_entropy_with_logits(logits, labels)
                val_loss_sum += float(vloss.item())
                val_batches  += 1
        inference_time_s += time.perf_counter() - t1
        val_loss_last     = val_loss_sum / max(val_batches, 1)
        epochs_ran        = epoch

        improved = (best_val_loss - val_loss_last) > EARLY_STOP_MIN_DELTA
        if improved:
            best_val_loss  = val_loss_last
            best_epoch     = epoch
            patience_left  = EARLY_STOP_PATIENCE
            best_state_cpu = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            if EARLY_STOP_ENABLED:
                patience_left -= 1
                if patience_left <= 0:
                    early_stopped = True
                    print(f'  early stop at epoch {epoch}')
                    break

        star = ' *' if improved else ''
        print(
            f'  epoch {epoch:02d} | train={train_loss_last:.5f} | val={val_loss_last:.5f} | '
            f'best_val={best_val_loss:.5f} | patience={patience_left}{star}'
        )

    if best_state_cpu is not None:
        model.load_state_dict(best_state_cpu)

    # ── Final inference ──────────────────────────────────────────────────────
    model.eval()
    probs_all = []
    t_inf = time.perf_counter()
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
            _     = batch.pop('labels')
            with autocast_ctx:
                logits = model(**batch).logits
            probs_all.append(torch.sigmoid(logits).float().detach().cpu().numpy())
    inference_time_s += time.perf_counter() - t_inf

    y_prob = np.vstack(probs_all)

    # ── Baseline metrics ─────────────────────────────────────────────────────
    y_pred_base = (y_prob >= 0.5).astype(np.int64)
    _, base_summary = multilabel_evaluation_report(y_val, y_pred_base, y_prob, LABEL_COLUMNS)

    # ── Tuned metrics ────────────────────────────────────────────────────────
    best_thresholds, thresholds_df = tune_per_label_thresholds(y_val, y_prob, LABEL_COLUMNS, THRESHOLD_GRID)
    y_pred_tuned  = apply_thresholds(y_prob, LABEL_COLUMNS, best_thresholds)
    per_label_df, tuned_summary = multilabel_evaluation_report(y_val, y_pred_tuned, y_prob, LABEL_COLUMNS)

    # ── Confusion matrices ───────────────────────────────────────────────────
    conf_pl_base_df, conf_agg_base_df   = make_confusion_artifacts(y_val, y_pred_base,  LABEL_COLUMNS)
    conf_pl_tuned_df, conf_agg_tuned_df = make_confusion_artifacts(y_val, y_pred_tuned, LABEL_COLUMNS)

    # ── Add sweep metadata columns ───────────────────────────────────────────
    for df in [per_label_df, thresholds_df, conf_pl_base_df, conf_pl_tuned_df]:
        df.insert(0, 'train_size',        spec['train_size'])
        df.insert(1, 'train_fraction',    spec['train_fraction'])
        df.insert(2, 'max_train_samples', spec['max_train_samples'])

    for df in [conf_agg_base_df, conf_agg_tuned_df]:
        df.insert(0, 'train_size',        spec['train_size'])
        df.insert(1, 'train_fraction',    spec['train_fraction'])
        df.insert(2, 'max_train_samples', spec['max_train_samples'])

    # ── Build summary row ────────────────────────────────────────────────────
    row = {
        'run_id':                  run_id,
        'train_size':              spec['train_size'],
        'train_fraction':          spec['train_fraction'],
        'max_train_samples':       spec['max_train_samples'],
        'actual_train_samples':    int(y_train.shape[0]),
        'actual_val_samples':      int(y_val.shape[0]),
        'validation_fraction':     VALIDATION_FRACTION,
        'random_state':            RANDOM_STATE,
        'use_iterative_stratify':  USE_ITERATIVE_STRATIFY,
        'rebalance_train':         REBALANCE_TRAIN,
        'model_name':              MODEL_NAME,
        'problem_type':            PROBLEM_TYPE,
        'max_length':              MAX_LENGTH,
        'batch_size':              BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'effective_batch_size':    BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        'max_epochs':              MAX_EPOCHS,
        'epochs_ran':              epochs_ran,
        'best_epoch':              best_epoch,
        'early_stopped':           early_stopped,
        'learning_rate':           LR,
        'weight_decay':            WEIGHT_DECAY,
        'warmup_ratio':            WARMUP_RATIO,
        'warmup_steps':            warmup_steps,
        'max_grad_norm':           MAX_GRAD_NORM,
        'early_stop_enabled':      EARLY_STOP_ENABLED,
        'early_stop_patience':     EARLY_STOP_PATIENCE,
        'early_stop_min_delta':    EARLY_STOP_MIN_DELTA,
        'loss_type':               'BCEWithLogitsLoss',
        'pos_weight_used':         USE_POS_WEIGHT,
        'mixed_precision':         'bf16',
        'use_amp_effective':       USE_AMP,
        'torch_compile':           USE_TORCH_COMPILE,
        'num_workers':             NUM_WORKERS,
        'num_params':              num_params,
        'train_time_s':            round(train_time_s, 2),
        'inference_time_s':        round(inference_time_s, 2),
        'best_val_loss':           best_val_loss,
        'train_loss_last':         train_loss_last,
        'val_loss_last':           val_loss_last,
        'baseline_micro_f1':       base_summary['f1_micro'],
        'baseline_macro_f1':       base_summary['f1_macro'],
        'baseline_samples_f1':     float(f1_score(y_val, y_pred_base,  average='samples', zero_division=0)),
        'tuned_micro_f1':          tuned_summary['f1_micro'],
        'tuned_macro_f1':          tuned_summary['f1_macro'],
        'tuned_samples_f1':        float(f1_score(y_val, y_pred_tuned, average='samples', zero_division=0)),
        'threshold_grid_start':    float(THRESHOLD_GRID[0]),
        'threshold_grid_end':      float(THRESHOLD_GRID[-1]),
        'threshold_grid_step':     round(float(THRESHOLD_GRID[1] - THRESHOLD_GRID[0]), 4),
    }

    # ── Save per-run files ───────────────────────────────────────────────────
    pd.DataFrame([row]).to_csv(row_file,           index=False)
    per_label_df.to_csv(per_label_file,            index=False)
    thresholds_df.to_csv(thresholds_file,          index=False)
    conf_pl_base_df.to_csv(conf_pl_base_file,      index=False)
    conf_pl_tuned_df.to_csv(conf_pl_tuned_file,    index=False)
    conf_agg_base_df.to_csv(conf_agg_base_file,    index=False)
    conf_agg_tuned_df.to_csv(conf_agg_tuned_file,  index=False)

    summary_rows.append(row)
    per_label_frames.append(per_label_df)
    threshold_frames.append(thresholds_df)
    conf_per_label_base_frames.append(conf_pl_base_df)
    conf_per_label_tuned_frames.append(conf_pl_tuned_df)
    conf_agg_base_rows.append(conf_agg_base_df.iloc[0].to_dict())
    conf_agg_tuned_rows.append(conf_agg_tuned_df.iloc[0].to_dict())

    print(
        f"  done | n={int(y_train.shape[0])} | "
        f"tuned_micro={tuned_summary['f1_micro']:.4f} | tuned_macro={tuned_summary['f1_macro']:.4f} | "
        f"time={train_time_s:.0f}s"
    )

print('Sweep complete.')

In [ ]:
# ── Consolidate and save master CSVs ─────────────────────────────────────────
summary_df     = pd.DataFrame(summary_rows).sort_values('train_size').reset_index(drop=True)
per_label_all  = pd.concat(per_label_frames,          ignore_index=True) if per_label_frames else pd.DataFrame()
thresholds_all = pd.concat(threshold_frames,          ignore_index=True) if threshold_frames else pd.DataFrame()
conf_pl_base_all  = pd.concat(conf_per_label_base_frames,  ignore_index=True) if conf_per_label_base_frames  else pd.DataFrame()
conf_pl_tuned_all = pd.concat(conf_per_label_tuned_frames, ignore_index=True) if conf_per_label_tuned_frames else pd.DataFrame()
conf_agg_base_all  = pd.DataFrame(conf_agg_base_rows)
conf_agg_tuned_all = pd.DataFrame(conf_agg_tuned_rows)

summary_df.to_csv(ARTIFACT_DIR        / 'bert_exp_06_sweep_summary.csv',               index=False)
per_label_all.to_csv(ARTIFACT_DIR     / 'bert_exp_06_sweep_per_label.csv',              index=False)
thresholds_all.to_csv(ARTIFACT_DIR    / 'bert_exp_06_sweep_thresholds.csv',             index=False)
conf_pl_base_all.to_csv(ARTIFACT_DIR  / 'bert_exp_06_sweep_confusion_per_label_baseline.csv', index=False)
conf_pl_tuned_all.to_csv(ARTIFACT_DIR / 'bert_exp_06_sweep_confusion_per_label_tuned.csv',    index=False)
conf_agg_base_all.to_csv(ARTIFACT_DIR / 'bert_exp_06_sweep_confusion_aggregate_baseline.csv', index=False)
conf_agg_tuned_all.to_csv(ARTIFACT_DIR / 'bert_exp_06_sweep_confusion_aggregate_tuned.csv',   index=False)

print(f'Saved consolidated CSVs to: {ARTIFACT_DIR}')
print(f'Runs: {len(summary_df)}')
summary_df[['train_size', 'actual_train_samples', 'epochs_ran', 'best_epoch',
            'tuned_micro_f1', 'tuned_macro_f1', 'train_time_s']].to_string(index=False)
summary_df

In [ ]:
# ── Graph 1: F1 vs training size ─────────────────────────────────────────────
plot_df = summary_df.sort_values('train_size').copy()

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(plot_df['train_size'], plot_df['baseline_micro_f1'],
        marker='o', linestyle='--', color='#aed6f1', linewidth=1.5, label='baseline micro F1 (t=0.5)')
ax.plot(plot_df['train_size'], plot_df['tuned_micro_f1'],
        marker='o', color='#2980b9', linewidth=2, label='tuned micro F1')
ax.plot(plot_df['train_size'], plot_df['tuned_macro_f1'],
        marker='s', color='#8e44ad', linewidth=2, label='tuned macro F1')

# DistilBERT reference lines
db_sizes  = sorted(DISTILBERT_MACRO_F1.keys())
db_macros = [DISTILBERT_MACRO_F1[s] for s in db_sizes]
db_micros = [DISTILBERT_MICRO_F1[s]  for s in db_sizes]
ax.plot(db_sizes, db_micros, marker='', linestyle=':', color='#2980b9', linewidth=1.2, alpha=0.55,
        label='DistilBERT tuned micro (ref)')
ax.plot(db_sizes, db_macros, marker='', linestyle=':', color='#8e44ad', linewidth=1.2, alpha=0.55,
        label='DistilBERT tuned macro (ref)')

ax.set_title(f'BERT Training Size Sweep — F1 vs Train Size\n(max_len={MAX_LENGTH}, lr={LR}, wd={WEIGHT_DECAY}, warmup={WARMUP_RATIO})')
ax.set_xlabel('Training samples')
ax.set_ylabel('F1 Score')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))
ax.set_ylim(0.4, 1.0)
ax.grid(alpha=0.3)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'bert_exp_06_graph_01_f1_vs_size.png', dpi=150)
plt.show()

In [ ]:
# ── Graph 2: Runtime vs training size ────────────────────────────────────────
plot_df = summary_df.sort_values('train_size').copy()

fig, ax = plt.subplots(figsize=(11, 4.5))

ax.plot(plot_df['train_size'], plot_df['train_time_s'],
        marker='o', color='#1a5276', linewidth=2, label='BERT train time (s)')
ax.plot(plot_df['train_size'], plot_df['inference_time_s'],
        marker='s', color='#117a65', linewidth=2, label='BERT inference time (s)')

# DistilBERT reference
db_sizes = sorted(DISTILBERT_TRAIN_TIME.keys())
db_times = [DISTILBERT_TRAIN_TIME[s] for s in db_sizes]
ax.plot(db_sizes, db_times, marker='', linestyle=':', color='#1a5276', linewidth=1.2, alpha=0.5,
        label='DistilBERT train time (ref)')

ax.set_title('BERT vs DistilBERT Runtime vs Training Size')
ax.set_xlabel('Training samples')
ax.set_ylabel('Seconds')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))
ax.grid(alpha=0.3)
ax.legend(fontsize=9)

# Annotate full-dataset point
full_row = plot_df.iloc[-1]
ax.annotate(
    f"full data\n{full_row['train_time_s']:.0f}s",
    (full_row['train_size'], full_row['train_time_s']),
    xytext=(-60, 12), textcoords='offset points',
    fontsize=8, color='#1a5276',
    arrowprops=dict(arrowstyle='->', color='#1a5276', lw=0.8),
)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'bert_exp_06_graph_02_runtime_vs_size.png', dpi=150)
plt.show()

In [ ]:
# ── Graph 3: Per-label F1 vs training size ───────────────────────────────────
if not per_label_all.empty:
    labels_order = list(LABEL_COLUMNS)
    plot_df      = summary_df.sort_values('train_size').copy()

    fig, ax = plt.subplots(figsize=(12, 5.5))

    for label in labels_order:
        label_df = per_label_all[per_label_all['label'] == label].sort_values('train_size')
        lw  = 2.2 if label in RARE_LABELS else 1.6
        ls  = '-' if label in RARE_LABELS else '--'
        ax.plot(
            label_df['train_size'], label_df['f1'],
            marker='o', markersize=4, linewidth=lw, linestyle=ls,
            color=LABEL_COLORS[label], label=label,
        )

    ax.set_title('BERT Per-Label F1 vs Training Size\n(solid = rare labels, dashed = common labels)')
    ax.set_xlabel('Training samples')
    ax.set_ylabel('Tuned F1')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))
    ax.set_ylim(0, 1.0)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9, ncol=2)

    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / 'bert_exp_06_graph_03_per_label_f1_vs_size.png', dpi=150)
    plt.show()

In [ ]:
# ── Graph 4: Rare-label F1 vs size with DistilBERT reference ─────────────────
if not per_label_all.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

    db_ref = {
        'threat':        DISTILBERT_THREAT_F1,
        'severe_toxic':  {10000: 0.5126, 20000: 0.5385, 30000: 0.5165, 40000: 0.5225,
                          50000: 0.5367, 60000: 0.5208, 70000: 0.5322, 80000: 0.5277,
                          90000: 0.5404, 100000: 0.5313, 110000: 0.5330, 120000: 0.5150,
                          130000: 0.5527, 140000: 0.5459, 143613: 0.5431},
        'identity_hate': {10000: 0.3148, 20000: 0.4870, 30000: 0.4548, 40000: 0.5703,
                          50000: 0.5862, 60000: 0.5655, 70000: 0.5935, 80000: 0.6073,
                          90000: 0.6154, 100000: 0.5871, 110000: 0.6186, 120000: 0.6036,
                          130000: 0.5895, 140000: 0.6296, 143613: 0.6082},
    }

    for i, label in enumerate(RARE_LABELS):
        label_df = per_label_all[per_label_all['label'] == label].sort_values('train_size')
        c = LABEL_COLORS[label]

        axes[i].plot(
            label_df['train_size'], label_df['f1'],
            marker='o', color=c, linewidth=2.2, label=f'BERT {label}',
        )

        if label in db_ref:
            db_sizes  = sorted(db_ref[label].keys())
            db_f1vals = [db_ref[label][s] for s in db_sizes]
            axes[i].plot(
                db_sizes, db_f1vals,
                marker='', linestyle=':', color=c, linewidth=1.4, alpha=0.55,
                label=f'DistilBERT {label} (ref)',
            )

        axes[i].set_title(label.replace('_', ' ').title(), color=c, fontsize=11)
        axes[i].set_xlabel('Training samples')
        axes[i].set_ylabel('Tuned F1')
        axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))
        axes[i].set_ylim(0, 1.0)
        axes[i].grid(alpha=0.3)
        axes[i].legend(fontsize=8)

    fig.suptitle('Rare-Label F1 vs Training Size: BERT vs DistilBERT (dotted)', fontsize=12)
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / 'bert_exp_06_graph_04_rare_label_f1_vs_size.png', dpi=150)
    plt.show()

In [ ]:
# ── Graph 5: Threshold evolution heatmap ─────────────────────────────────────
# Shows how optimal per-label thresholds change as training size increases.
# Stable thresholds signal well-calibrated model probabilities.
if not thresholds_all.empty:
    labels_order = list(LABEL_COLUMNS)
    sizes_sorted = sorted(thresholds_all['train_size'].unique())

    heatmap_data = pd.DataFrame(index=sizes_sorted, columns=labels_order, dtype=float)
    for size in sizes_sorted:
        sub = thresholds_all[thresholds_all['train_size'] == size].set_index('label')
        for label in labels_order:
            if label in sub.index:
                heatmap_data.loc[size, label] = sub.loc[label, 'best_threshold']

    fig, ax = plt.subplots(figsize=(11, 7))
    sns.heatmap(
        heatmap_data.astype(float),
        annot=True, fmt='.3f', cmap='RdYlGn_r',
        vmin=0.05, vmax=0.90,
        linewidths=0.4, linecolor='white',
        ax=ax,
        xticklabels=[l.replace('_', '\n') for l in labels_order],
        yticklabels=[f'{int(s/1000)}k' for s in sizes_sorted],
    )
    ax.set_title(
        'Optimal Decision Threshold per Label vs Training Size\n'
        '(low threshold = rare/poorly-calibrated label; stable rows = calibration converged)',
        fontsize=11,
    )
    ax.set_xlabel('Label')
    ax.set_ylabel('Training samples')

    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / 'bert_exp_06_graph_05_threshold_heatmap.png', dpi=150)
    plt.show()

In [ ]:
# ── Graph 6: Macro F1 tuning gain vs training size ───────────────────────────
# Tuning gain = tuned_macro_f1 - baseline_macro_f1.
# If gain shrinks with more data, the model becomes better-calibrated at t=0.5.
plot_df = summary_df.sort_values('train_size').copy()
plot_df['tuning_gain_macro'] = plot_df['tuned_macro_f1'] - plot_df['baseline_macro_f1']
plot_df['tuning_gain_micro'] = plot_df['tuned_micro_f1'] - plot_df['baseline_micro_f1']

fig, ax = plt.subplots(figsize=(11, 4.5))

ax.bar(
    plot_df['train_size'] - 2200, plot_df['tuning_gain_macro'],
    width=4200, color='#8e44ad', alpha=0.8, label='macro F1 gain (tuned - baseline)',
)
ax.bar(
    plot_df['train_size'] + 2200, plot_df['tuning_gain_micro'],
    width=4200, color='#2980b9', alpha=0.6, label='micro F1 gain (tuned - baseline)',
)
ax.axhline(0, color='black', linewidth=0.8)

ax.set_title(
    'Threshold Tuning Gain (tuned \u2212 baseline F1) vs Training Size\n'
    '(macro gain dominated by rare labels; higher gain = model more miscalibrated at t=0.5)'
)
ax.set_xlabel('Training samples')
ax.set_ylabel('\u0394 F1 (tuned \u2212 baseline)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'bert_exp_06_graph_06_tuning_gain_vs_size.png', dpi=150)
plt.show()

In [ ]:
# ── Graph 7: BERT vs DistilBERT — per-label F1 and ROC-AUC at full dataset ───
if not per_label_all.empty:
    labels_order = list(LABEL_COLUMNS)
    full_size    = summary_df['train_size'].max()
    full_df      = per_label_all[per_label_all['train_size'] == full_size].set_index('label')

    bert_f1  = [full_df.loc[l, 'f1']      for l in labels_order]
    bert_roc = [full_df.loc[l, 'roc_auc'] for l in labels_order]
    db_f1    = [DISTILBERT_FULL_F1[l]     for l in labels_order]
    db_roc   = [DISTILBERT_FULL_ROC[l]    for l in labels_order]
    delta_f1 = [b - d for b, d in zip(bert_f1,  db_f1)]
    delta_roc= [b - d for b, d in zip(bert_roc, db_roc)]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    x = np.arange(len(labels_order))
    w = 0.35

    # F1 panel
    axes[0].bar(x - w/2, db_f1,   width=w, label='DistilBERT (128 tok, Exp F)', color='#aed6f1', edgecolor='white')
    axes[0].bar(x + w/2, bert_f1, width=w, label='BERT (192 tok, Exp 06)',       color='#1a5276', edgecolor='white')
    for i, (d, b, g) in enumerate(zip(db_f1, bert_f1, delta_f1)):
        col  = '#117a65' if g >= 0 else '#922b21'
        sign = '+' if g >= 0 else ''
        axes[0].text(x[i] + w/2, b + 0.01, f'{sign}{g:.3f}', ha='center', va='bottom', fontsize=8, color=col)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([l.replace('_', '\n') for l in labels_order])
    axes[0].set_ylim(0, 1.0)
    axes[0].set_ylabel('Tuned F1')
    axes[0].set_title('Per-Label F1: BERT vs DistilBERT')
    axes[0].legend(fontsize=9)
    axes[0].grid(axis='y', alpha=0.3)

    # ROC-AUC panel
    axes[1].bar(x - w/2, db_roc,   width=w, label='DistilBERT', color='#aed6f1', edgecolor='white')
    axes[1].bar(x + w/2, bert_roc, width=w, label='BERT',       color='#1a5276', edgecolor='white')
    for i, (d, b, g) in enumerate(zip(db_roc, bert_roc, delta_roc)):
        col  = '#117a65' if g >= 0 else '#922b21'
        sign = '+' if g >= 0 else ''
        axes[1].text(x[i] + w/2, b + 0.0002, f'{sign}{g:.4f}', ha='center', va='bottom', fontsize=7.5, color=col)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([l.replace('_', '\n') for l in labels_order])
    axes[1].set_ylim(0.97, 1.0)
    axes[1].set_ylabel('ROC-AUC')
    axes[1].set_title('Per-Label ROC-AUC: BERT vs DistilBERT')
    axes[1].legend(fontsize=9)
    axes[1].grid(axis='y', alpha=0.3)

    fig.suptitle(
        f'BERT (max_len=192, {int(full_size/1000)}k samples) vs DistilBERT (max_len=128, 143.6k samples)',
        fontsize=12,
    )
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / 'bert_exp_06_graph_07_bert_vs_distilbert_full.png', dpi=150)
    plt.show()

    # Save delta CSV
    delta_df = pd.DataFrame({
        'label':          labels_order,
        'distilbert_f1':  [round(v, 4) for v in db_f1],
        'bert_f1':        [round(v, 4) for v in bert_f1],
        'delta_f1':       [round(g, 4) for g in delta_f1],
        'distilbert_roc': [round(v, 4) for v in db_roc],
        'bert_roc':       [round(v, 4) for v in bert_roc],
        'delta_roc':      [round(g, 4) for g in delta_roc],
    })
    delta_df.to_csv(ARTIFACT_DIR / 'bert_exp_06_vs_distilbert.csv', index=False)
    delta_df

In [ ]:
# ── Graph 8: Efficiency frontier — macro F1 vs training time ─────────────────
# Plots the Pareto curve for each model: one point per training size.
# Points to the upper-left are more efficient (same F1, less time).
plot_df = summary_df.sort_values('train_size').copy()

fig, ax = plt.subplots(figsize=(10, 6))

# BERT sweep points
sc_bert = ax.scatter(
    plot_df['train_time_s'], plot_df['tuned_macro_f1'],
    c='#1a5276', s=80, zorder=5, label='BERT (this sweep)', edgecolors='white', linewidth=0.8,
)
ax.plot(plot_df['train_time_s'], plot_df['tuned_macro_f1'],
        color='#1a5276', linewidth=1.2, alpha=0.5)

# DistilBERT reference points
db_sizes  = sorted(DISTILBERT_TRAIN_TIME.keys())
db_times  = [DISTILBERT_TRAIN_TIME[s] for s in db_sizes]
db_macros = [DISTILBERT_MACRO_F1[s]   for s in db_sizes]
sc_db = ax.scatter(
    db_times, db_macros,
    c='#117a65', s=60, zorder=4, marker='D', label='DistilBERT (Exp F)', edgecolors='white', linewidth=0.8,
)
ax.plot(db_times, db_macros, color='#117a65', linewidth=1.2, alpha=0.5)

# Annotate size labels for BERT
for _, row in plot_df.iterrows():
    size_k = int(row['train_size'] / 1000)
    if size_k % 20 == 0 or row['train_size'] == plot_df['train_size'].max():
        ax.annotate(
            f"{size_k}k",
            (row['train_time_s'], row['tuned_macro_f1']),
            xytext=(6, 4), textcoords='offset points',
            fontsize=7.5, color='#1a5276',
        )

ax.set_title(
    'Efficiency Frontier: Tuned Macro F1 vs Training Time\n'
    '(upper-left = better efficiency; each point = one training-size run)'
)
ax.set_xlabel('Training time (seconds)')
ax.set_ylabel('Tuned macro F1')
ax.grid(alpha=0.3)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'bert_exp_06_graph_08_efficiency_frontier.png', dpi=150)
plt.show()

In [ ]:
# ── Summary tables ────────────────────────────────────────────────────────────
import IPython.display as ipd

key_cols = [
    'train_size', 'train_fraction', 'actual_train_samples',
    'epochs_ran', 'best_epoch', 'early_stopped',
    'baseline_micro_f1', 'baseline_macro_f1',
    'tuned_micro_f1',    'tuned_macro_f1',
    'train_time_s', 'inference_time_s',
]

print('=== Sweep Summary ===')
ipd.display(summary_df[key_cols].round(4))

print('\n=== Full-dataset per-label metrics ===')
full_size = summary_df['train_size'].max()
ipd.display(
    per_label_all[per_label_all['train_size'] == full_size]
    [['label', 'precision', 'recall', 'f1', 'roc_auc']]
    .round(4)
    .reset_index(drop=True)
)

print('\n=== Full-dataset optimal thresholds ===')
ipd.display(
    thresholds_all[thresholds_all['train_size'] == full_size]
    [['label', 'best_threshold', 'best_f1_on_val']]
    .reset_index(drop=True)
)